https://pmc.ncbi.nlm.nih.gov/articles/PMC12605207/



## NAFLD - Adolescent (Zhang et al., 2025)
**Reference**: Zhang et al. "From traditional metabolic markers to ensemble learning: 
comparative application of machine learning models for predicting NAFLD risk in adolescents." 
Frontiers in Endocrinology, 2025.

**Label**: ALT > 26 IU/L (males) or > 22 IU/L (females), excluding hepatitis B/C positive

**Inclusion criteria**: Adolescents aged 11-20, no viral hepatitis

**Deviations from reference study**:
- Reference restricted to NHANES 2011-2020 (n=2,132); we include all available cycles (n=6,141)
- Prevalence consistent with reference (15.7% vs 15.4%)
- 5-fold stratified CV instead of train/test split
- No SMOTE applied

In [6]:
import pandas as pd
import numpy as np
import os, pickle

full = pd.read_pickle("/Users/alva/Documents/nhanes_tasks/data/NHANES/NHANES_master.pkl")

In [7]:
# --- NAFLD task ---
# Reference: Zhang et al. (2025), Frontiers in Endocrinology
# Population: US adolescents aged 11-20, NHANES 2011-2020
# Label: ALT > 26 IU/L (males) or > 22 IU/L (females), no viral hepatitis
# Deviation: 5-fold stratified CV instead of train/test split, no SMOTE

subset_nafld = full[
    (full['RIDAGEYR'] >= 11) &
    (full['RIDAGEYR'] <= 20)
].copy()

subset_nafld = subset_nafld[
    (subset_nafld['LBXHBS'] != 1) &
    (subset_nafld['LBXHCR'] != 1)
]

subset_nafld['NAFLD'] = np.where(
    (subset_nafld['RIAGENDR'] == 1) & (subset_nafld['LBXSATSI'] > 26), 1,
    np.where(
        (subset_nafld['RIAGENDR'] == 2) & (subset_nafld['LBXSATSI'] > 22), 1, 0
    )
)

features_nafld = ['BMXWAIST', 'LBXSTR', 'LBXPLTSI', 'BMXHT', 'LBXSGL',
                  'LBXWBCSI', 'LBXSCH', 'LBXRBCSI', 'LBDHDD', 'RIDAGEYR', 'RIAGENDR']

data_nafld = subset_nafld[features_nafld + ['NAFLD']].dropna()
X_nafld = data_nafld[features_nafld].values
y_nafld = data_nafld['NAFLD'].values

print(f"NAFLD: n={len(y_nafld)}, prevalence={y_nafld.mean():.3f}, features={X_nafld.shape[1]}")

NAFLD: n=6141, prevalence=0.157, features=11


In [8]:
os.makedirs('/Users/alva/Documents/nhanes_tasks/data/tasks/', exist_ok=True)

task = {
    'X': X_nafld,
    'y': y_nafld,
    'features': features_nafld,
    'name': 'nafld'
}

with open('/Users/alva/Documents/nhanes_tasks/data/tasks/nafld.pkl', 'wb') as f:
    pickle.dump(task, f)

print("Saved.")

Saved.
